In [1]:
import pandas as pd
### Get thes pam data
import os
BASE_DIR = os.path.dirname(os.path.abspath("__vsc_ipynb_file_"))
SPAM_DATA = os.path.join(BASE_DIR, 'data_set', 'SMSSpamCollection.csv')

In [2]:
messages = pd.read_csv(SPAM_DATA, sep="\t", names=["lable", "messages"])

In [3]:
messages

,lable,messages
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [10]:
messages.shape
for i in messages['messages'][0:10]:
    print(i)

Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Ok lar... Joking wif u oni...
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
U dun say so early hor... U c already then say...
Nah I don't think he goes to usf, he lives around here though
FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv
Even my brother is not like to speak with me. They treat me like aids patent.
As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune
WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.
Had your mobile 11 months or more? U R entitl

In [5]:
#Data cleaning and preprocessing
import re
import nltk

In [13]:
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\suganth\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [20]:
corpus = []
for message in messages['messages']:
    review = re.sub('[^a-zA-Z0-9]', " ", message)
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if word not in stopwords.words("english")]
    review = " ".join(review)
    corpus.append(review)

In [43]:
# Creating BoW
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2500, binary=True, ngram_range=(2,2))
X=cv.fit_transform(corpus).toarray()

In [47]:
y=pd.get_dummies(messages['lable'])
y=y.iloc[:,1].values

In [48]:
y

array([False, False,  True, ..., False, False, False], shape=(5572,))

In [50]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.20, random_state=0)

##### The Purpose of train_test split
When we train a machine learning model, we don’t want to test it on the same data it learned from — that would give us an unrealistically high accuracy.
So we split the dataset into:

Training set → used to teach the model.

Test set → used to evaluate how well the model generalizes to unseen data.

That’s exactly what train_test_split from sklearn.model_selection does: it randomly divides your dataset into training and testing portions.

test_size=0.2 → 20% of data goes to testing, 80% to training.

random_state=0 → ensures reproducibility (same split every time).

In [51]:
from sklearn.naive_bayes import  MultinomialNB
spam_detect_model = MultinomialNB().fit(X_train, y_train)

In [52]:
# prediction
y_pred=spam_detect_model.predict(X_test)

In [54]:
from sklearn.metrics import accuracy_score, classification_report

In [55]:
score=accuracy_score(y_test, y_pred)
print(score)

0.9713004484304932


#### Assignment 
Creat the above model using TF-IDF